In [2]:
%pip install pandas numpy scikit-learn joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier

from sklearn.naive_bayes import MultinomialNB

from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [7]:
df = pd.read_csv(
    "../data/spam.csv",
    encoding="latin-1"
)

df = df[["v1", "v2"]]

df.columns = [
    "Email Type",
    "Email Text"
]

df.head()

,Email Type,Email Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
df["Email Type"] = df["Email Type"].map({
    "ham": 0,
    "spam": 1
})

df.head()

,Email Type,Email Text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
print(
    df["Email Type"]
    .value_counts()
)

Email Type
0    4825
1     747
Name: count, dtype: int64


In [10]:
X = df["Email Text"]

y = df["Email Type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1,2)
)

X_train_vec = vectorizer.fit_transform(
    X_train
)

X_test_vec = vectorizer.transform(
    X_test
)

print(X_train_vec.shape)
print(X_test_vec.shape)

(4457, 5000)
(1115, 5000)


In [12]:
models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000
        ),

    "Naive Bayes":
        MultinomialNB(),

    "Decision Tree":
        DecisionTreeClassifier(
            max_depth=20,
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=50,
            random_state=42
        ),

    "Linear SVM":
        LinearSVC()
}

In [14]:
results = []

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(
        X_train_vec,
        y_train
    )

    predictions = model.predict(
        X_test_vec
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions
    )

    recall = recall_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions
    )

    results.append({

        "Model": name,

        "Accuracy":
            round(accuracy * 100, 2),

        "Precision":
            round(precision * 100, 2),

        "Recall":
            round(recall * 100, 2),

        "F1 Score":
            round(f1 * 100, 2)
    })

results_df = pd.DataFrame(
    results
)

results_df

Training Logistic Regression...
Training Naive Bayes...
Training Decision Tree...
Training Random Forest...
Training Linear SVM...


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,96.77,97.50,78.00,86.67
1,Naive Bayes,97.22,100.00,79.33,88.48
2,Decision Tree,95.70,91.80,74.67,82.35
3,Random Forest,97.49,98.41,82.67,89.86
4,Linear SVM,98.03,97.76,87.33,92.25


In [15]:
results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
4,Linear SVM,98.03,97.76,87.33,92.25
3,Random Forest,97.49,98.41,82.67,89.86
1,Naive Bayes,97.22,100.00,79.33,88.48
0,Logistic Regression,96.77,97.50,78.00,86.67
2,Decision Tree,95.70,91.80,74.67,82.35


In [16]:
import json

metrics = results_df.to_dict(
    orient="records"
)

with open(
    "../data/model_metrics.json",
    "w"
) as file:

    json.dump(
        metrics,
        file,
        indent=4
    )

print(
    "model_metrics.json created successfully"
)

model_metrics.json created successfully


In [17]:
best_model = results_df.iloc[0]

print("Best Model:")
print(best_model["Model"])

print("\nAccuracy:")
print(best_model["Accuracy"])

Best Model:
Linear SVM

Accuracy:
98.03
